# 🪟 Модуль 2. Урок 5: Оконные функции (Window Functions) в деталях

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_05_window_functions_advanced.ipynb)

---

## 1. Главные оконные функции аналитика данных

1. `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`: Уникальная нумерация строк внутри группы (идеально для дедупликации).
2. `DENSE_RANK() OVER (...)`: Ранжирование без пропусков (Топ-3 товара в каждой категории).
3. `LAG(col, 1) OVER (...)` и `LEAD(col, 1) OVER (...)`: Доступ к предыдущей или следующей строке (расчет темпов роста MoM / YoY).
4. `SUM(col) OVER (ORDER BY ...)`: Расчет нарастающего итога (Cumulative Revenue).


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
WITH monthly_revenue AS (
    SELECT 
        SUBSTR(o.order_date, 1, 7) AS ym,
        ROUND(SUM(oi.total_amount), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Completed'
    GROUP BY ym
)
SELECT 
    ym,
    revenue,
    -- Нарастающий итог с начала года
    ROUND(SUM(revenue) OVER (ORDER BY ym), 2) AS running_total,
    -- Доход за предыдущий месяц
    LAG(revenue, 1) OVER (ORDER BY ym) AS prev_month,
    -- Относительный прирост месяц-к-месяцу (MoM %)
    ROUND((revenue - LAG(revenue, 1) OVER (ORDER BY ym)) * 100.0 / 
          LAG(revenue, 1) OVER (ORDER BY ym), 2) AS mom_growth_pct
FROM monthly_revenue
ORDER BY ym;
"""

print("ФИНАНСОВАЯ АНАЛИТИКА ЧЕРЕЗ ОКОННЫЕ ФУНКЦИИ:")
print(pd.read_sql(query, conn))
